In [3]:
import pandas as pd

# Membaca file dengan nama yang sesuai di panel kiri (menggunakan spasi)
df_raw = pd.read_csv("IMDB Dataset.csv")

# Mengambil subset 2.000 baris agar ringan dijalankan (Sesuai Bab 3.4.1) [cite: 394, 395]
df = df_raw.sample(n=2000, random_state=42).reset_index(drop=True)

print("Dataset dari komputer Anda berhasil dimuat!")
print(df.head())

Dataset dari komputer Anda berhasil dimuat!
                                              review sentiment
0  I really liked this Summerslam due to the look...  positive
1  Not many television shows appeal to quite as m...  positive
2  The film quickly gets to a major chase scene w...  negative
3  Jane Austen would definitely approve of this o...  positive
4  Expectations were somewhat high for me when I ...  negative


In [4]:
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# STEP 1: PRA-PEMROSESAN TEKS & LABEL ENCODING
def clean_text(text):
    text = text.lower()  # 1. Mengubah ke huruf kecil (lowercase) [cite: 414]
    text = re.sub(r'<br\s*/?>', ' ', text)  # 2. Menghapus tag HTML break line dari teks review
    text = re.sub(r'[^a-z\s]', '', text)  # 3. Menghapus angka dan tanda baca [cite: 397]
    text = re.sub(r'\s+', ' ', text).strip()  # 4. Merapikan spasi ganda
    return text

# Terapkan pembersihan pada kolom review teks
df['cleaned_review'] = df['review'].apply(clean_text)

# Label encoding: positive menjadi 1, negative menjadi 0 [cite: 416]
df['label'] = df['sentiment'].map({'positive': 1, 'negative': 0})


# STEP 2: MEMBAGI DATASET
X_text = df['cleaned_review']
y = df['label']

# Membagi data menjadi 80% Data Latih (Train) dan 20% Data Uji (Test) [cite: 426]
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=42
)

# STEP 3: VEKTORISASI TF-IDF
# Membatasi fitur kata unik maksimal 5000 agar performa stabil [cite: 419]
tfidf = TfidfVectorizer(max_features=5000)
X_train = tfidf.fit_transform(X_train_text).toarray()  # Fitting data latih [cite: 420]
X_test = tfidf.transform(X_test_text).toarray()        # Transformasi data uji

# STEP 4: PELATIHAN MODEL NAIVE BAYES (Sesuai Bab 3.2 & 3.4.1 Poin 5)
# Menggunakan MultinomialNB karena sangat optimal untuk klasifikasi teks [cite: 358, 360]
model_nb = MultinomialNB()
model_nb.fit(X_train, y_train)

# Melakukan prediksi pada data uji [cite: 366]
y_pred_nb = model_nb.predict(X_test)

# STEP 5: EVALUASI PERFORMA MODEL
print("=== HASIL EVALUASI NAIVE BAYES ===")
print("Akurasi Model:", accuracy_score(y_test, y_pred_nb))  # Menampilkan skor akurasi [cite: 368]
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_nb))  # Menampilkan confusion matrix [cite: 434]
print("\nClassification Report:\n", classification_report(y_test, y_pred_nb))  # Menampilkan presisi, recall, f1-score

=== HASIL EVALUASI NAIVE BAYES ===
Akurasi Model: 0.805

Confusion Matrix:
 [[192  17]
 [ 61 130]]

Classification Report:
               precision    recall  f1-score   support

           0       0.76      0.92      0.83       209
           1       0.88      0.68      0.77       191

    accuracy                           0.81       400
   macro avg       0.82      0.80      0.80       400
weighted avg       0.82      0.81      0.80       400



In [5]:
from sklearn.linear_model import LogisticRegression

# 1. Inisialisasi dan Pelatihan Model Regresi Logistik (Sesuai Bab 3.3)
model_lr = LogisticRegression()
model_lr.fit(X_train, y_train)

# 2. Prediksi terhadap data uji
y_pred_lr = model_lr.predict(X_test)

# 3. Evaluasi Performa Model
print("=== HASIL EVALUASI REGRESI LOGISTIK ===")
print("Akurasi Model Regresi Logistik:", accuracy_score(y_test, y_pred_lr))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred_lr))
print("\nClassification Report:\n", classification_report(y_test, y_pred_lr))

=== HASIL EVALUASI REGRESI LOGISTIK ===
Akurasi Model Regresi Logistik: 0.8375

Confusion Matrix:
 [[177  32]
 [ 33 158]]

Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.85      0.84       209
           1       0.83      0.83      0.83       191

    accuracy                           0.84       400
   macro avg       0.84      0.84      0.84       400
weighted avg       0.84      0.84      0.84       400



In [6]:
def tes_ulasan_baru(kalimat_input):
    # 1. Bersihkan teks input menggunakan fungsi clean_text yang sudah dibuat sebelumnya
    kalimat_bersih = clean_text(kalimat_input)

    # 2. Transformasikan teks menjadi vektor numerik menggunakan TF-IDF yang sudah dilatih
    vektor_input = tfidf.transform([kalimat_bersih]).toarray()

    # 3. Lakukan prediksi menggunakan kedua model
    pred_nb = model_nb.predict(vektor_input)[0]
    pred_lr = model_lr.predict(vektor_input)[0]

    # Mapping output angka ke teks sentimen
    label_map = {1: "POSITIF", 0: "NEGATIF"}

    print(f"Ulasan: \"{kalimat_input}\"")
    print(f"-> Prediksi Naive Bayes     : {label_map[pred_nb]}")
    print(f"-> Prediksi Regresi Logistik: {label_map[pred_lr]}")
    print("-" * 50)

# Silakan ganti atau tambah kalimat di bawah ini sesuka Anda!
tes_ulasan_baru("The cinematography was brilliant and the acting was top-tier. Highly recommended!")
tes_ulasan_baru("What a waste of time. The plot makes no sense and the characters are so flat.")
tes_ulasan_baru("The movie was okay, but the ending felt rushed and a bit disappointing.")

Ulasan: "The cinematography was brilliant and the acting was top-tier. Highly recommended!"
-> Prediksi Naive Bayes     : POSITIF
-> Prediksi Regresi Logistik: POSITIF
--------------------------------------------------
Ulasan: "What a waste of time. The plot makes no sense and the characters are so flat."
-> Prediksi Naive Bayes     : NEGATIF
-> Prediksi Regresi Logistik: NEGATIF
--------------------------------------------------
Ulasan: "The movie was okay, but the ending felt rushed and a bit disappointing."
-> Prediksi Naive Bayes     : NEGATIF
-> Prediksi Regresi Logistik: NEGATIF
--------------------------------------------------
